# Day 9: Transformer Architecture - Positional Encoding & Multi-Head Attention

## Introduction
Welcome to Day 9! Today, we transition from the high-level LLM fundamentals into the core architecture that powers modern AI: the **Transformer**. As an engineer, you know that understanding the underlying mechanisms of a system is crucial for debugging and optimization. 

We will focus on two foundational blocks:
1.  **Positional Encoding:** Injecting sequence order information into our data.
2.  **Multi-Head Attention:** Allowing the model to focus on different parts of the input sequence simultaneously.

## Core Theory (Just-in-Time)

### 1. The "Why" and "How" of Positional Encoding
**Why:** Traditional RNNs process tokens sequentially, inherently knowing which token came first. Transformers process all tokens simultaneously (in parallel). Without positional encoding, a Transformer would treat "The dog bit the man" and "The man bit the dog" as the exact same input. We must mathematically inject the *position* of each token into its embedding.

**How:** We add a unique positional vector to each token's embedding vector. The original paper ("Attention Is All You Need") uses sine and cosine functions of different frequencies. 
* Even dimensions get a sine wave.
* Odd dimensions get a cosine wave.
This continuous mathematical formulation allows the model to easily learn relative positions, even for sequences longer than those seen during training.

### 2. The "Why" and "How" of Multi-Head Attention
**Why:** Single attention (which you built yesterday) calculates how much focus one token should give to all others based on *one* set of learned parameters. However, tokens relate to each other in multiple ways (e.g., grammatical relationship, semantic meaning, pronoun resolution). We need the model to capture multiple *types* of relationships simultaneously.

**How:** Instead of calculating attention once, we calculate it $h$ times in parallel ($h$ = number of "heads"). 
1. We project our input into $h$ different sets of Query (Q), Key (K), and Value (V) matrices.
2. We compute scaled dot-product attention for each head independently.
3. We concatenate the results from all heads.
4. We pass the concatenated output through a final linear projection to merge the insights.



## Code Implementation: Positional Encoding

Below is a production-grade Python implementation of Positional Encoding using standard math libraries. We avoid using external heavy frameworks to ensure you understand the core logic. Notice the strict type hinting and docstrings.


In [ ]:
import math
from typing import List

def get_positional_encoding(seq_len: int, d_model: int) -> List[List[float]]:
    """
    Generates the positional encoding matrix for a given sequence length and model dimension.
    
    Args:
        seq_len: The length of the input sequence (number of tokens).
        d_model: The dimension of the embedding vector for each token.
        
    Returns:
        A 2D list (matrix) of shape (seq_len, d_model) representing the positional encodings.
    """
    pe: List[List[float]] = []
    
    for pos in range(seq_len):
        pos_embedding: List[float] = []
        for i in range(d_model):
            # Calculate the denominator term: 10000^(2i/d_model)
            # Since i is the dimension index, we map it to pairs (2i, 2i+1)
            # To handle both even and odd indices elegantly in one loop:
            exponent: float = (i // 2 * 2) / d_model
            denominator: float = math.pow(10000.0, exponent)
            
            value: float = pos / denominator
            
            if i % 2 == 0:
                pos_embedding.append(math.sin(value))
            else:
                pos_embedding.append(math.cos(value))
                
        pe.append(pos_embedding)
        
    return pe

# --- Example Usage ---
if __name__ == "__main__":
    sequence_length = 5
    embedding_dimension = 4
    
    pos_encoding_matrix = get_positional_encoding(seq_len=sequence_length, d_model=embedding_dimension)
    
    print("Positional Encoding Matrix (5 tokens, 4 dimensions):")
    for row in pos_encoding_matrix:
        formatted_row = [f"{val: .4f}" for val in row]
        print(f"[{', '.join(formatted_row)}]")



## Code Implementation: Multi-Head Attention Helper Functions

To build Multi-Head Attention purely in Python, we first need some matrix math utilities. In a production setting, this is heavily optimized by libraries like NumPy or PyTorch, but writing them from scratch hardens your understanding of the tensor shapes involved.


In [ ]:
from typing import List, Tuple

def matrix_multiply(A: List[List[float]], B: List[List[float]]) -> List[List[float]]:
    """Multiplies two matrices A (m x n) and B (n x p) to produce C (m x p)."""
    m = len(A)
    n = len(A[0])
    p = len(B[0])
    
    C = [[0.0 for _ in range(p)] for _ in range(m)]
    for i in range(m):
        for j in range(p):
            for k in range(n):
                C[i][j] += A[i][k] * B[k][j]
    return C

def transpose(A: List[List[float]]) -> List[List[float]]:
    """Transposes matrix A (m x n) to (n x m)."""
    return [[A[j][i] for j in range(len(A))] for i in range(len(A[0]))]

def softmax_row(row: List[float]) -> List[float]:
    """Applies softmax to a single 1D list (row)."""
    max_val = max(row)
    exp_vals = [math.exp(x - max_val) for x in row]  # subtract max for numerical stability
    sum_exp = sum(exp_vals)
    return [x / sum_exp for x in exp_vals]

def matrix_softmax(A: List[List[float]]) -> List[List[float]]:
    """Applies softmax to each row of a matrix independently."""
    return [softmax_row(row) for row in A]

def scale_matrix(A: List[List[float]], scalar: float) -> List[List[float]]:
    """Multiplies every element in matrix A by a scalar value."""
    return [[val * scalar for val in row] for row in A]

def concatenate_matrices_horizontally(matrices: List[List[List[float]]]) -> List[List[float]]:
    """
    Concatenates a list of matrices horizontally.
    Example: two (N x d) matrices become one (N x 2d) matrix.
    """
    num_rows = len(matrices[0])
    result = []
    for r in range(num_rows):
        combined_row = []
        for mat in matrices:
            combined_row.extend(mat[r])
        result.append(combined_row)
    return result



## Code Implementation: The Multi-Head Attention Block

Now we assemble the components into the Multi-Head Attention logic.


In [ ]:
def scaled_dot_product_attention(Q: List[List[float]], 
                                 K: List[List[float]], 
                                 V: List[List[float]]) -> List[List[float]]:
    """
    Computes Scaled Dot-Product Attention for a single head.
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k))V
    """
    d_k = len(K[0])
    
    # 1. Q * K^T
    K_T = transpose(K)
    scores = matrix_multiply(Q, K_T)
    
    # 2. Scale by 1/sqrt(d_k)
    scale_factor = 1.0 / math.sqrt(d_k)
    scaled_scores = scale_matrix(scores, scale_factor)
    
    # 3. Softmax
    attention_weights = matrix_softmax(scaled_scores)
    
    # 4. Multiply by V
    output = matrix_multiply(attention_weights, V)
    
    return output

def multi_head_attention_simulation(
    input_sequence: List[List[float]], 
    num_heads: int,
    d_model: int
) -> List[List[float]]:
    """
    Simulates Multi-Head Attention by splitting the input into multiple heads.
    
    In a real implementation, we would multiply the input by learned weight matrices (W_q, W_k, W_v)
    for each head. To keep this example focused on the architecture flow without external libraries
    and random weight generation, we will partition the input features directly among the heads.
    
    Args:
        input_sequence: Matrix of shape (seq_len, d_model) representing our embedded tokens + positional encoding.
        num_heads: Number of attention heads.
        d_model: The total embedding dimension.
        
    Returns:
        The output matrix of shape (seq_len, d_model) after multi-head attention processing.
    """
    if d_model % num_heads != 0:
        raise ValueError("d_model must be perfectly divisible by num_heads.")
        
    seq_len = len(input_sequence)
    d_k = d_model // num_heads  # Dimension per head
    
    head_outputs: List[List[List[float]]] = []
    
    for h in range(num_heads):
        # SIMULATION: For this head, extract a specific slice of the d_model features.
        # This acts as our Q, K, and V projections for this specific head.
        start_idx = h * d_k
        end_idx = start_idx + d_k
        
        Q_head = [[row[i] for i in range(start_idx, end_idx)] for row in input_sequence]
        K_head = [[row[i] for i in range(start_idx, end_idx)] for row in input_sequence]
        V_head = [[row[i] for i in range(start_idx, end_idx)] for row in input_sequence]
        
        # Compute attention for this specific head
        head_out = scaled_dot_product_attention(Q_head, K_head, V_head)
        head_outputs.append(head_out)
        
    # Concatenate all heads together to get back to (seq_len, d_model)
    final_output = concatenate_matrices_horizontally(head_outputs)
    
    # In a full model, we would now multiply final_output by a learned output weight matrix W_o.
    return final_output

# --- Example Usage ---
if __name__ == "__main__":
    # Let's say we have 3 tokens, and an embedding dimension of 6
    dummy_input = [
        [1.0, 0.5, 0.2, 0.1, 0.8, 0.9], # Token 1
        [0.2, 0.9, 0.1, 0.5, 0.1, 0.4], # Token 2
        [0.8, 0.1, 0.6, 0.9, 0.2, 0.3], # Token 3
    ]
    
    # Process through 2 heads. Each head will operate on 3 dimensions (6 / 2)
    mha_output = multi_head_attention_simulation(dummy_input, num_heads=2, d_model=6)
    
    print("\nMulti-Head Attention Output (3 tokens, 6 dimensions):")
    for row in mha_output:
        formatted_row = [f"{val: .4f}" for val in row]
        print(f"[{', '.join(formatted_row)}]")



## Common Pitfalls in Production

When transitioning from this fundamental code to production systems (like using vLLM or HuggingFace), watch out for these pitfalls:

1.  **Context Window Overflows:** Transformers process attention across the entire sequence length $N$. The memory and compute cost of self-attention scales quadratically, $O(N^2)$. Feeding excessive context without summarization or chunking will cause Out-Of-Memory (OOM) errors on your GPUs.
2.  **Dimension Mismatch Errors:** The most common bug when building custom transformer blocks is tensor shape mismatch. Always aggressively assert that `d_model % num_heads == 0`.
3.  **Ignoring Masking:** In the decoder portion of a transformer (used in models like GPT), you *must* apply a causal mask to the attention scores before the softmax step. If you forget this, the model "looks into the future" during training and fails catastrophically during inference.
4.  **Floating Point Instability:** In our from-scratch code, large values in the dot-product before softmax can cause exponential explosion. That is *why* we divide by $\sqrt{d_k}$. Always ensure your scaling factor is correctly applied when modifying attention logic.

---

## Practical Lab / Homework

**Your Task for Today:**

In the multi-head attention simulation above, we simply sliced the input features to simulate different heads. However, real multi-head attention uses learned weight matrices ($W_Q$, $W_K$, $W_V$) to project the input into different sub-spaces.

Your assignment is to complete the `real_multi_head_projection` function below. You will take the input sequence and multiply it by a provided Weight Matrix to create the Query (Q) vectors for a *single* head. 

*Constraint:* Use the `matrix_multiply` utility function provided earlier in the notebook. Do not use external libraries.


In [ ]:
def real_multi_head_projection(
    input_sequence: List[List[float]], 
    weight_matrix_Q: List[List[float]]
) -> List[List[float]]:
    """
    Projects the input sequence into the Query space for a single attention head.
    
    Args:
        input_sequence: Matrix of shape (seq_len, d_model)
        weight_matrix_Q: Learned projection matrix of shape (d_model, d_k)
        
    Returns:
        The Query matrix Q of shape (seq_len, d_k)
    """
    
    # YOUR CODE HERE
    # Hint: You need to perform a matrix multiplication between the input_sequence and weight_matrix_Q
    
    # Replace the empty list with your working logic. Do not leave this as a stub.
    Q_projected = matrix_multiply(input_sequence, weight_matrix_Q)
    return Q_projected

# --- Test your implementation ---
if __name__ == "__main__":
    # 2 tokens, 4 embedding dimensions
    test_seq = [
        [1.0, 2.0, 0.5, 0.1],
        [0.5, 1.0, 2.0, 0.5]
    ]
    
    # 4 embedding dimensions projected down to 2 dimensions for this head (d_k = 2)
    test_Wq = [
        [0.1, 0.2],
        [0.3, 0.1],
        [0.1, 0.5],
        [0.8, 0.1]
    ]
    
    try:
        result = real_multi_head_projection(test_seq, test_Wq)
        print("Lab Output - Projected Query Matrix:")
        for row in result:
             print([f"{val: .4f}" for val in row])
             
        # Expected shape: 2x2
        assert len(result) == 2, "Result should have 2 rows"
        assert len(result[0]) == 2, "Result should have 2 columns"
        print("Success! Dimensions match expected output.")
    except Exception as e:
        print(f"Error in implementation: {e}")

